# ξ_F: is there BAO in the geometry of the filament network?

**Question.** The Fil sample is the Full sample cut by a region of space: the tubes of radius $R_{\rm fil}$ around the
DisPerSE spines. Does that region *itself* (empty or not) carry a BAO feature?

**The mask.** With $d(\mathbf x)$ the distance from a point $\mathbf x$ to the nearest spine,

$$F(\mathbf x)=\begin{cases}1 & d_{\rm lo}\le d(\mathbf x)\le d_{\rm hi}\\ 0 & \text{otherwise}\end{cases}$$

For Fil, $(d_{\rm lo}, d_{\rm hi}) = (0, 3)\,h^{-1}$Mpc, the same interval `split_vars` uses for the galaxies. By construction

$$\boxed{\;n_{\rm Fil}(\mathbf x)=F(\mathbf x)\,n_{\rm Full}(\mathbf x)\;}$$

with $n$ the galaxy number density. $F$ lights every point inside the tubes, with or without galaxies, so it is not built
from the galaxies whose ξ we compare it with.

**What is measured.** With $f=\langle F\rangle$ the filling fraction (the fraction of the box volume inside the tubes),
the overdensity of the mask and its two-point functions are

$$\delta_F(\mathbf x)=\frac{F(\mathbf x)}{f}-1,\qquad
\boxed{\;\xi_F(r)=\langle\delta_F(\mathbf x)\,\delta_F(\mathbf x+\mathbf r)\rangle\;},\qquad
P_F(k)=\langle|\tilde\delta_F(\mathbf k)|^2\rangle/V$$

$V=L^3$ is the box volume and $\tilde\delta_F$ the Fourier transform of $\delta_F$. $F$ is a field, not a set of points, so
there is **no shot noise**.

**What it is for.** A diagnostic only: nothing here goes into the fits or the covariance. $F$ cannot be used as a window
in the model ($1+\xi_{\rm Fil}=(1+\xi_F)(1+\xi_{\rm Full})$ needs $F$ independent of the galaxies, and the spines are
traced from them).

**Inputs.** The DisPerSE skeleton of the box (segments), the catalogue (for the validation in section 2), the monopoles
and spectra of the run.  **Output:** `data/processed/exploratory/xi_F/`.

In [ ]:
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt
from scipy import fft
from scipy.spatial import cKDTree

REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))       # notebooks/exploratory -> repo
sys.path.insert(0, os.path.join(REPO, 'src'))
import bao_fitting as bf
from multidark_data_loader import load_catalog, select_sample
from multidark_power import _kgrid                    # k grid of the FFT (same convention as the spectra)

# ---------------- settings (change here only) ----------------
L        = bf.L_BOX                      # box side, h^-1 Mpc
MAG      = -21.2                         # magnitude cut of the run to compare with
BINSEP   = 5.0                           # monopole bin width, h^-1 Mpc (5.0 = baseline)
SEP      = (5.0, 200.0)                  # separation range of the monopole files
TUBES    = {'Fil': (0.0, 3.0), 'NonFil': (10.0, 100.0)}   # (d_lo, d_hi), as split_vars of multidark_analysis.py

SKELETON_FILE   = os.path.join(REPO, 'data', 'raw', 'skeleton_segs.txt')   # <- DisPerSE skeleton of the box
SKELETON_FORMAT = 'segs_ascii'           # 'segs_ascii' (skelconv -to segs_ascii) or 'npy' (N x 6 array U, V)

NG       = 384                           # grid per side (cell L/NG = 2.6 h^-1 Mpc); rerun with 512 to check convergence
DS_SPINE = 1.0                           # spacing of the points sampled along each segment, h^-1 Mpc
D_MAX    = max(hi for _, hi in TUBES.values())   # distances beyond this are not needed (stored as inf)
N_VALID  = 200_000                       # galaxies used to validate the skeleton against dist_fil
FIT_AMP  = (40.0, 70.0)                  # range where each xi's amplitude B^2 is matched to the template (comparison plots)
FIT_RANGE = (50.0, 150.0)                # range of the broadband fit of the comparison plots
KBAND    = (0.02, 0.30)                  # k range of the Fourier-space wiggle plot, h/Mpc
KUSE     = 0.6                           # P_F is trusted up to KUSE x the grid Nyquist frequency
# --------------------------------------------------------------

RAW = os.path.join(REPO, 'data', 'raw')
MONO_DIR = os.path.join(REPO, 'data', 'processed', 'monopoles', 'box')
PK_FILE = os.path.join(REPO, 'data', 'processed', 'pk', 'box', f'pk_XISMU_box_mag{MAG:.1f}_dist_fil.npz')
OUT_DIR = os.path.join(REPO, 'data', 'processed', 'exploratory', 'xi_F')
os.makedirs(OUT_DIR, exist_ok=True)

def tube_key(lo, hi):
    # label of the subsample with lo <= d_fil <= hi in the monopole / spectra files
    return f"{lo:.1f}leqd_mathrmfilleq{hi:.1f}"

def monopole_file(key):
    return os.path.join(MONO_DIR, f"mag={MAG}_sep={SEP[0]}-{SEP[1]}_binsep={BINSEP}_{key}.npz")

print(f"repo {REPO}\ngrid {NG}^3, cell {L/NG:.2f} h^-1 Mpc, tubes {TUBES}")

## 1. Skeleton and distance to the spines

The skeleton is a list of segments, each with end points $\mathbf U$ and $\mathbf V$. Points are placed along every segment
every `DS_SPINE`, and $d(\mathbf x)$ is the distance to the nearest of them (periodic box). For a point at distance $d$ from
a segment this overestimates $d$ by at most $\sqrt{d^2+(\Delta s/2)^2}-d$, with $\Delta s$ = `DS_SPINE`: 0.04 $h^{-1}$Mpc at
$d=3$ for $\Delta s=1$.

A segment that crosses the box edge is taken with the shortest periodic vector between its end points.
To make the input file from a DisPerSE skeleton: `skelconv <file>.NDskl -to segs_ascii`.

In [ ]:
def load_skeleton(path, fmt):
    # Segment end points U, V as two (n_seg, 3) arrays, in h^-1 Mpc
    if fmt == 'segs_ascii':
        a = np.loadtxt(path, comments='#', usecols=range(6))
    elif fmt == 'npy':
        a = np.load(path)
    else:
        raise ValueError(f"unknown skeleton format {fmt}")
    return a[:, :3], a[:, 3:6]


def spine_points(U, V, ds, L):
    # Points every <= ds along each segment (both end points included), wrapped into [0, L)
    D = V - U
    D -= L * np.round(D / L)                                   # shortest periodic vector
    n = np.maximum(np.ceil(np.linalg.norm(D, axis=1) / ds).astype(int), 1)
    seg = np.repeat(np.arange(len(U)), n + 1)
    t = np.concatenate([np.linspace(0.0, 1.0, m + 1) for m in n])
    return np.mod(U[seg] + t[:, None] * D[seg], L)


def distance_grid(tree, L, ng, d_max, slab=8):
    # d(x) at the centres of the ng^3 cells (float32); cells farther than d_max get inf
    c = (np.arange(ng) + 0.5) * L / ng
    d = np.empty((ng, ng, ng), np.float32)
    yy, zz = np.meshgrid(c, c, indexing='ij')
    yz = np.column_stack([yy.ravel(), zz.ravel()])
    for i0 in range(0, ng, slab):
        xs = c[i0:i0 + slab]
        pts = np.column_stack([np.repeat(xs, len(yz)), np.tile(yz, (len(xs), 1))])
        dist, _ = tree.query(pts, k=1, distance_upper_bound=d_max, workers=-1)
        d[i0:i0 + len(xs)] = dist.reshape(len(xs), ng, ng)
    return d

In [ ]:
U, V = load_skeleton(SKELETON_FILE, SKELETON_FORMAT)
seg_len = np.linalg.norm((V - U) - L * np.round((V - U) / L), axis=1)
P_sp = spine_points(U, V, DS_SPINE, L)
tree = cKDTree(P_sp, boxsize=L)
print(f"{len(U)} segments, total length {seg_len.sum():.3e} h^-1 Mpc "
      f"(median segment {np.median(seg_len):.2f}, max {seg_len.max():.2f});  {len(P_sp)} spine points")
print(f"coordinate range: {min(U.min(), V.min()):.2f} - {max(U.max(), V.max()):.2f}  (box 0 - {L:g})")

## 2. Validation: does this skeleton give the catalogue's `dist_fil`?

The tubes must be the region the Fil galaxies were selected from. For `N_VALID` random galaxies the distance to the
spines is recomputed and compared with `dist_fil` from `mock_withfilament.csv`. They should agree to about `DS_SPINE`/2 or
better, and the Fil / not-Fil classification should agree for (nearly) every galaxy. If they don't (wrong file, units, a
different distance definition), stop here: the rest of the notebook would describe a different selection.

In [ ]:
cat = select_sample(load_catalog(os.path.join(RAW, 'to_mock.csv'), os.path.join(RAW, 'mock_withfilament.csv')), MAG)
rng = np.random.default_rng(1)
iv = rng.choice(len(cat), size=min(N_VALID, len(cat)), replace=False)
pos_v = np.mod(cat[['x', 'y', 'z']].values[iv], L)
d_cat = cat['dist_fil'].values[iv]
d_new, _ = tree.query(pos_v, k=1, workers=-1)

diff = d_new - d_cat
near = d_cat <= 2 * D_MAX
print(f"{len(iv)} galaxies (mag < {MAG}):  d_new - d_cat  median {np.median(diff):+.3f}, "
      f"16-84% [{np.percentile(diff, 16):+.3f}, {np.percentile(diff, 84):+.3f}], "
      f"|diff| > 0.5: {np.mean(np.abs(diff) > 0.5):.4f}")
for name, (lo, hi) in TUBES.items():
    a = (d_cat >= lo) & (d_cat <= hi); b = (d_new >= lo) & (d_new <= hi)
    print(f"  {name:<7} {lo}-{hi}: in catalogue {a.sum()}, recomputed {b.sum()}, "
          f"classification agrees for {np.mean(a == b):.4f} of the galaxies")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
m = d_cat < 15
ax[0].plot(d_cat[m], d_new[m], ',', alpha=0.3); ax[0].plot([0, 15], [0, 15], 'k--', lw=0.8)
ax[0].set(xlabel='dist_fil (catalogue)', ylabel='recomputed distance', xlim=(0, 15), ylim=(0, 15))
ax[1].hist(np.clip(diff[m], -2, 2), bins=100); ax[1].set(xlabel='recomputed - catalogue (clipped at ±2)', yscale='log')
plt.tight_layout(); plt.show()

## 3. The mask on the grid

$d$ is evaluated at the centre of each of the `NG`³ cells (cached in `OUT_DIR`, it takes a few minutes), and
$F = 1$ where $d_{\rm lo}\le d\le d_{\rm hi}$. Printed per tube: the filling fraction $f$, and the fraction of the
subsample's galaxies whose cell is lit. That fraction is below 1 only because a cell is judged by its centre (tube radius
3 vs cell 2.6 $h^{-1}$Mpc); the BAO scale is far from the cell scale.
The slice shows $F$ (grey) and the Fil galaxies (dots) in one cell-thick slab.

In [ ]:
cache = os.path.join(OUT_DIR, f"dgrid_{os.path.basename(SKELETON_FILE).rsplit('.', 1)[0]}_ng{NG}_ds{DS_SPINE}_dmax{D_MAX:g}.npy")
if os.path.exists(cache):
    dgrid = np.load(cache)
else:
    t0 = time.time()
    dgrid = distance_grid(tree, L, NG, D_MAX)
    np.save(cache, dgrid)
    print(f"distance grid: {time.time() - t0:.0f} s -> {cache}")

pos_all = np.mod(cat[['x', 'y', 'z']].values, L)
cell = np.floor(pos_all / (L / NG)).astype(int) % NG
masks = {}
for name, (lo, hi) in TUBES.items():
    F = (dgrid >= lo) & (dgrid <= hi)
    masks[name] = F
    sel = (cat['dist_fil'].values >= lo) & (cat['dist_fil'].values <= hi)
    lit = F[cell[sel, 0], cell[sel, 1], cell[sel, 2]]
    print(f"{name:<7} {lo}-{hi}: filling fraction f = {F.mean():.4f};  {sel.sum()} galaxies, "
          f"{lit.mean():.3f} of them in lit cells;  galaxies per lit cell {sel.sum() / F.sum():.3f} "
          f"(box average {len(cat) / NG**3:.3f})")

i_slab = 0
xmax = 250.0
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(masks['Fil'][:, :, i_slab].T, origin='lower', extent=(0, L, 0, L), cmap='Greys', vmin=0, vmax=2)
lo, hi = TUBES['Fil']
s_ = (cat['dist_fil'].values >= lo) & (cat['dist_fil'].values <= hi) & (cell[:, 2] == i_slab)
ax.plot(pos_all[s_, 0], pos_all[s_, 1], '.', ms=2, color='C3')
ax.set(xlim=(0, xmax), ylim=(0, xmax), xlabel='x [h$^{-1}$Mpc]', ylabel='y [h$^{-1}$Mpc]',
       title=f'F (Fil tubes) and Fil galaxies, slab z < {L / NG:.1f}')
plt.show()

## 4. $P_F(k)$ and $\xi_F(r)$

With $\tilde\delta_F(\mathbf k)$ the discrete Fourier transform of $\delta_F$ on the grid ($N_c=$ `NG`³ cells):

$$P_F(k)=\frac{V}{N_c^2}\,\big\langle|\tilde\delta_F(\mathbf k)|^2\big\rangle_{\rm shell},\qquad
\xi_F(\mathbf r)=\frac{1}{N_c}\sum_{\mathbf x}\delta_F(\mathbf x)\,\delta_F(\mathbf x+\mathbf r)
=\frac{1}{N_c}\,\mathrm{IFFT}\big[|\tilde\delta_F|^2\big](\mathbf r)$$

$\xi_F(\mathbf r)$ is computed on the grid directly (no Hankel transform) and averaged over the cells whose separation
$|\mathbf r|$ falls in each bin of the monopole files. $P_F$ uses the same $k$ shells as `multidark_power` (width $2\pi/L$),
up to `KUSE` × the Nyquist frequency. $F$ is sampled at cell centres, so there is no mass-assignment window to deconvolve;
aliasing grows near Nyquist, hence the `KUSE` cut.

**Error band (indicative).** The Gaussian formula of the pipeline with $P_F$ and no shot noise,
$C_{ij}=\frac{2}{V^2}\sum_{\mathbf k}P_F(k)^2\langle j_0\rangle_i\langle j_0\rangle_j$, where $\langle j_0\rangle_i$ is the spherical
Bessel function $j_0(kr)=\sin(kr)/(kr)$ averaged over bin $i$. $F$ is very non-Gaussian, so this only shows the size of the
cosmic-variance scatter; it is not used for any test.

In [ ]:
def field_stats(F, L, s_edges, kuse=KUSE):
    # P_F(k) in shells of width 2 pi/L up to kuse x Nyquist, and xi_F in the bins s_edges
    ng = F.shape[0]; Nc = ng**3; V = L**3
    f = F.mean()
    D = fft.rfftn(F.astype(np.float32) / f - 1.0, workers=-1)
    A = (D.real**2 + D.imag**2).astype(np.float32)
    del D
    # P(k)
    kx, ky, kz = (x.astype(np.float32) for x in _kgrid(L, ng))
    K = np.sqrt(kx**2 + ky**2 + kz**2)
    w = np.full(K.shape, 2.0, np.float32); w[..., 0] = 1.0
    if ng % 2 == 0:
        w[..., -1] = 1.0                                       # rfft: count each +-k pair once
    kf = 2 * np.pi / L
    edges_k = kf * (0.5 + np.arange(int(kuse * np.pi * ng / L / kf)))
    ib = np.digitize(K, edges_k) - 1
    ok = (ib >= 0) & (ib < len(edges_k) - 1)
    nm = np.bincount(ib[ok], weights=w[ok], minlength=len(edges_k) - 1)
    km = np.bincount(ib[ok], weights=(K * w)[ok], minlength=len(edges_k) - 1) / np.maximum(nm, 1)
    Pk = np.bincount(ib[ok], weights=(A * w)[ok], minlength=len(edges_k) - 1) / np.maximum(nm, 1) * V / Nc**2
    del K, ib, ok, w
    # xi(r): inverse FFT of |delta_k|^2, then shell average over the cells with |r| in each bin
    xi3 = fft.irfftn(A, s=(ng,) * 3, workers=-1) / Nc
    del A
    m = int(np.ceil(s_edges[-1] / (L / ng))) + 1
    idx = np.arange(-m, m + 1)
    sub = xi3[np.ix_(idx % ng, idx % ng, idx % ng)]
    del xi3
    r1 = idx * L / ng
    R = np.sqrt(r1[:, None, None]**2 + r1[None, :, None]**2 + r1[None, None, :]**2)
    jb = np.digitize(R, s_edges) - 1
    okr = (jb >= 0) & (jb < len(s_edges) - 1)
    cnt = np.bincount(jb[okr], minlength=len(s_edges) - 1)
    xi = np.bincount(jb[okr], weights=sub[okr], minlength=len(s_edges) - 1) / np.maximum(cnt, 1)
    good = nm > 0
    return dict(f=f, k=km[good], P=Pk[good], nmodes=nm[good], kmax=edges_k[-1], xi=xi, var=float(sub[m, m, m]))


def gaussian_cov_field(s_edges, k, P, L, kmax):
    # Gaussian covariance of the xi monopole of a field without shot noise, summed over the box's modes up to kmax
    kk, cnt = bf.lattice_mode_counts(L, kmax)
    J = bf.shell_avg_j0(kk, s_edges[:-1], s_edges[1:])
    Pm = np.interp(kk, k, P)
    return (2.0 / L**6) * (J * (cnt * Pm**2)) @ J.T


s_ref, _, _ = bf.load_monopole(monopole_file('full'))
s_edges = bf.edges_from_centres(s_ref)
res = {}
for name, F in masks.items():
    t0 = time.time()
    r = field_stats(F, L, s_edges)
    r['cov'] = gaussian_cov_field(s_edges, r['k'], r['P'], L, r['kmax'])
    res[name] = r
    print(f"{name:<7} f = {r['f']:.4f}, variance of delta_F = {r['var']:.2f} (expected (1-f)/f = {(1 - r['f']) / r['f']:.2f}), "
          f"P_F to k = {r['kmax']:.2f} h/Mpc  [{time.time() - t0:.0f} s]")

## 5. Comparison with the galaxy monopoles and the template

Every curve gets the same treatment (unweighted least squares; a picture, not a fit result):

1. Amplitude $B^2$: $\xi$ matched to the template $\xi(s;\alpha=1,\Sigma_{\rm th})$ on `FIT_AMP`, below the BAO
   ($\Sigma_{\rm th}$ = theory damping, `bf.sigma_nl_theory`).
2. Smooth model with that amplitude fixed and the pipeline's broadband fitted on `FIT_RANGE`:

$$\xi_{\rm nw\,model}(s)=B^2\,\xi_{\rm nw}(s)+a_0+a_1\frac{100}{s}+a_2\Big(\frac{100}{s}\Big)^2,$$

($\xi_{\rm nw}$ = the no-wiggle part of `bf.BAOTemplate`, bin-averaged; $a_i$ = broadband coefficients,
`bf.broadband_columns`). $B^2$ is fixed first because over 50–150 $h^{-1}$Mpc the smooth $\xi_{\rm nw}$ is nearly a
combination of the broadband terms, so a free $B^2$ is not determined there.

* Top: $s^2\xi/B^2$ (shapes on a common scale).
* Bottom: $s^2(\xi-\xi_{\rm nw\,model})/B^2$, what the smooth model cannot absorb. A BAO feature is a bump at $s\approx r_d$
  (dotted line). The black line is the same residual for the template $\xi(s;\alpha=1,\Sigma_{\rm th})$, i.e. what a
  BAO of the theory damping looks like after this procedure. The band on the mask is the indicative Gaussian error of
  section 4.

In [ ]:
lin = bf.linear_theory()
kT, P_w, P_nw = bf.generate_templates(lin=lin)
T = bf.BAOTemplate(kT, P_w, P_nw, lin)
Sig_th = bf.sigma_nl_theory(lin)['Sigma_mono']
s = s_ref
xi_bao = T.xi(Sig_th, edges=s_edges)
xi_nw = T.xi_nw(edges=s_edges)
mA = (s >= FIT_AMP[0]) & (s <= FIT_AMP[1])
mF = (s >= FIT_RANGE[0]) & (s <= FIT_RANGE[1])
X = bf.broadband_columns(s)

def nowiggle_fit(xi):
    # B^2 from FIT_AMP, then B^2 xi_nw + broadband fitted on FIT_RANGE
    B2 = float(xi[mA] @ xi_bao[mA] / (xi_bao[mA] @ xi_bao[mA]))
    c, *_ = np.linalg.lstsq(X[mF], (xi - B2 * xi_nw)[mF], rcond=None)
    return B2, B2 * xi_nw + X @ c

for name, (lo, hi) in TUBES.items():
    curves = {f'mask F ({name} tubes)': (res[name]['xi'], np.sqrt(np.diag(res[name]['cov'])))}
    for lab, key in ((f'{name} galaxies', tube_key(lo, hi)), ('Full galaxies', 'full')):
        if os.path.exists(monopole_file(key)):
            curves[lab] = (bf.load_monopole(monopole_file(key))[1], None)
        else:
            print(f"missing {monopole_file(key)}")
    curves['template'] = (xi_bao, None)
    fig, ax = plt.subplots(2, 1, figsize=(7, 7), sharex=True, gridspec_kw=dict(height_ratios=(3, 2)))
    for i, (lab, (xi, err)) in enumerate(curves.items()):
        B2, mod = nowiggle_fit(xi)
        print(f"{name}: {lab:<22} B^2 = {B2:.4g}")
        col, kw = ('k', dict(lw=1)) if lab == 'template' else (f'C{i}', dict(marker='o', ms=3))
        lab_ = rf'template ($\Sigma$ = {Sig_th:.1f})' if lab == 'template' else lab
        ax[0].plot(s[mF], (s**2 * xi / B2)[mF], '-', color=col, label=lab_, **kw)
        y1 = s**2 * (xi - mod) / B2
        ax[1].plot(s[mF], y1[mF], '-', color=col, **kw)
        if err is not None:
            e = s**2 * err / abs(B2)
            ax[1].fill_between(s[mF], (y1 - e)[mF], (y1 + e)[mF], color=col, alpha=0.2, lw=0)
    ax[1].axhline(0, color='k', lw=0.5)
    for a in ax:
        a.axvline(lin['rd'], color='grey', ls=':', lw=0.8)
    ax[0].set(ylabel=r'$s^2\xi/B^2$', title=f'{name}: mask vs galaxies ($B^2$ from {FIT_AMP}, broadband on {FIT_RANGE})')
    ax[1].set(xlabel=r's [$h^{-1}$Mpc]', ylabel=r'$s^2(\xi-\xi_{\rm nw\,model})/B^2$')
    ax[0].legend(fontsize=8)
    plt.tight_layout(); plt.show()

**Fourier space.** The same idea: $P(k)$ divided by $P_{\rm nw}(k)$ times a smooth polynomial
$c_0+c_1k+\dots+c_4k^4$ fitted to $P/P_{\rm nw}$ on `KBAND` (least squares), so the broadband shape of each spectrum is
removed and only oscillations remain. Black: the template with the theory damping,
$[P_{\rm nw}+(P_{\rm lin}-P_{\rm nw})\,e^{-k^2\Sigma_{\rm th}^2/2}]/P_{\rm nw}$. The galaxy spectra come from the run's spectra file
and include shot noise (a constant, absorbed by the polynomial up to its $k$ dependence relative to $P_{\rm nw}$).

In [ ]:
spec = bf.load_spectra(PK_FILE) if os.path.exists(PK_FILE) else None
Pnw = lambda k: np.interp(k, kT, P_nw)
Pdamp = lambda k: np.interp(k, kT, P_nw + (P_w - P_nw) * np.exp(-kT**2 * Sig_th**2 / 2))

def wiggles(k, P):
    # P / (P_nw x polynomial of degree 4 in k), polynomial fitted to P/P_nw in KBAND
    m = (k >= KBAND[0]) & (k <= KBAND[1])
    r = P[m] / Pnw(k[m])
    c = np.polynomial.polynomial.polyfit(k[m], r, 4)
    return k[m], r / np.polynomial.polynomial.polyval(k[m], c)

fig, axs = plt.subplots(1, len(TUBES), figsize=(6 * len(TUBES), 4), squeeze=False)
for ax, (name, (lo, hi)) in zip(axs[0], TUBES.items()):
    ax.plot(*wiggles(res[name]['k'], res[name]['P']), '-', color='C0', label=f'mask F ({name})')
    if spec is not None:
        for lab, key, c in ((f'{name} galaxies', tube_key(lo, hi), 'C1'), ('Full galaxies', 'full', 'C2')):
            if key in spec['labels']:
                i = spec['labels'].index(key)
                ax.plot(*wiggles(spec['k'], spec['P'][i, i]), '-', color=c, lw=1, label=lab)
    kk = np.linspace(*KBAND, 400)
    ax.plot(kk, Pdamp(kk) / Pnw(kk), 'k-', lw=1, label=rf'template ($\Sigma$ = {Sig_th:.1f})')
    ax.axhline(1, color='k', lw=0.5)
    ax.set(xlabel=r'k [h/Mpc]', ylabel=r'$P/(P_{\rm nw}\times{\rm poly})$', title=name)
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
out = os.path.join(OUT_DIR, f"xi_F_ng{NG}_ds{DS_SPINE}_binsep{BINSEP}.npz")
save = dict(s=s, s_edges=s_edges, tubes=np.array([[*TUBES[n]] for n in TUBES]), names=np.array(list(TUBES)),
            skeleton=SKELETON_FILE, NG=NG, DS_SPINE=DS_SPINE, L=L)
for name, r in res.items():
    for key in ('f', 'k', 'P', 'nmodes', 'kmax', 'xi', 'cov'):
        save[f'{name}_{key}'] = r[key]
np.savez(out, **save)
print(f"saved -> {out}")